# Reproducing Howard's GENIE/NuWro `W`-split $Q^2$ ratio plot

Howard's technote shows GENIE/NuWro as a function of true $Q^2$, split into "All W", "W < 1.4 GeV" and "W > 1.4 GeV" -- a diagnostic for whether the two generators agree on the CC-other/CC-inclusive composition, and whether that agreement depends on the hadronic invariant mass regime.

**Important population caveat (read before trusting the numbers below):** this notebook does **not** reproduce Howard's comparison on equal footing. His plot (and our own `BackgroungTemplates.ipynb` derivation) compares two *bare generator-truth* samples -- no detector simulation, no selection, just `flagCCINC`/`flagCC0pi` truth flags from NUISANCE flat trees. Here, per instruction, **GENIE is taken from `1muNp0pi_Nge1_uncontained.root`** -- our own file, already run through full detector simulation and the medulla selection -- while **NuWro is still the bare NUISANCE flat trees**. That means the GENIE side is conditioned on reconstructing something resembling a 1$\mu$N$p$0$\pi$ candidate topology, which the NuWro side is not. Any true CC-other event that never gets reconstructed into one of the trees used below (`selected`, `sideband`, `signal`) is invisible to the GENIE numerator and denominator alike, but NuWro counts it. So a difference in shape from Howard's plot can come from selection efficiency, not just generator physics -- see the Interpretation cell at the end, which shows this is exactly what happens.

**Population definitions, checked directly against the files (not assumed):**
- CC-inclusive: `true_cc == 1` in our file; `flagCCINC` in the NuWro flat trees. Checked category-by-category in `1muNp0pi_Nge1_uncontained.root`: categories 0-7 are 100% `true_cc==1`, category 8 is 100% `true_cc==0` (NC), category 9 is `true_cc` NaN (no true interaction) -- so `true_cc==1` is an exact CC-inclusive flag, not a guess built from category numbers whose meaning wasn't otherwise confirmed.
- CC-other: `category` in {4,5,6,7} on our side (same categories targeted by `parameterSet_backgroundFit_true_q2.yaml`'s `applyCondition`); `flagCCINC & ~flagCC0pi` on the NuWro side (same proxy used in `BackgroungTemplates.ipynb`).
- W split at 1.4 GeV: `true_generator_w` on our side (confirmed <0.03% NaN within CC-inclusive here); `W_nuc_rest` on the NuWro side (confirmed <0.04% NaN within CC-inclusive) -- both are nucleon-rest-frame invariant hadronic mass, same physical definition, from independent generator truth.
- Units, confirmed empirically: NuWro's `Q2_true` and `W_nuc_rest` are MeV$^2$/MeV (`/1e6` and `/1000` respectively to get GeV$^2$/GeV); our own `true_generator_q2`/`true_generator_w` are already GeV$^2$/GeV.
- No flux or generator-tune weighting is applied on either side -- this is meant to isolate a Q$^2$/W-shape comparison, and adding our own `ppfx_cv_weight` (a NuMI flux correction) would inject a flux-model correction into what's supposed to be an interaction-generator comparison.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import json
import uproot

ICARUS_ROOT = '/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/My Drive/🏛 PhD Repository/🚀 Research/🤖 Experiments&Projects/ICARUS'

BACKGROUND_CATEGORIES = [4, 5, 6, 7]
Q2_EDGES = np.arange(0.0, 10.5, 1.0)   # 10 bins, [0,1),[1,2),...,[9,10) GeV^2 -- matches Howard's plot axis
W_SPLIT = 1.4  # GeV

def open_with_retry(path, tries=5, sleep_s=2.0):
    """The Google-Drive-mounted files occasionally hit a transient FUSE-level
    'Resource deadlock avoided' OSError on first read -- retrying a few times
    with a short sleep clears it. Same helper as in BackgroungTemplates.ipynb."""
    last_exc = None
    for _ in range(tries):
        try:
            return uproot.open(path)
        except OSError as exc:
            last_exc = exc
            time.sleep(sleep_s)
    raise last_exc


## Load GENIE -- our own file, `events/nominal/{selected,sideband,signal}` combined

These are the three trees in `1muNp0pi_Nge1_uncontained.root` that actually carry `true_generator_q2`/`true_generator_w` (confirmed directly: the broader `preselection` tree, 19.7M entries, does not have either branch at all).

In [ ]:
SELECTION_ROOT = f'{ICARUS_ROOT}/ICARUS_CC0pi_Selection/data'
f_genie = open_with_retry(f'{SELECTION_ROOT}/1muNp0pi_Nge1_uncontained.root')

branches_g = ['true_category', 'true_cc', 'true_generator_q2', 'true_generator_w']
g_data = {b: [] for b in branches_g}
for treename in ['events/nominal/selected', 'events/nominal/sideband', 'events/nominal/signal']:
    arrs = f_genie[treename].arrays(branches_g, library='np')
    for b in branches_g:
        g_data[b].append(arrs[b])
g_data = {b: np.concatenate(v) for b, v in g_data.items()}

g_ccinc = (g_data['true_cc'] == 1)
g_ccother = g_ccinc & np.isin(g_data['true_category'], BACKGROUND_CATEGORIES)
g_q2 = g_data['true_generator_q2']
g_w = g_data['true_generator_w']

print(f"GENIE (selected+sideband+signal): {len(g_q2)} rows, "
      f"CC-inclusive={g_ccinc.sum()}, CC-other={g_ccother.sum()}")
print(f"  true_generator_w NaN frac within CC-inclusive: {np.mean(~np.isfinite(g_w[g_ccinc])):.5f}")


## Load NuWro -- bare generator NUISANCE flat trees (all 10 files, same set used in `BackgroungTemplates.ipynb`)

In [ ]:
GENERATORS_DIR = f'{ICARUS_ROOT}/ICARUS_CC0pi_GUNDAM/data/Generators'

n_q2_list, n_ccother_list, n_ccinc_list, n_w_list = [], [], [], []
for i in range(10):
    path = f'{GENERATORS_DIR}/NuWro/fhc_Nu14/output_NuWro_{i}.nuisflat.root'
    t = open_with_retry(path)['FlatTree_VARS']
    arrs = t.arrays(['Q2_true', 'flagCCINC', 'flagCC0pi', 'W_nuc_rest'], library='np')
    ccinc = arrs['flagCCINC'].astype(bool)
    cc0pi = arrs['flagCC0pi'].astype(bool)
    n_q2_list.append(arrs['Q2_true'] / 1e6)        # MeV^2 -> GeV^2
    n_w_list.append(arrs['W_nuc_rest'] / 1000.0)   # MeV -> GeV
    n_ccother_list.append(ccinc & ~cc0pi)
    n_ccinc_list.append(ccinc)

n_q2 = np.concatenate(n_q2_list)
n_w = np.concatenate(n_w_list)
n_ccother = np.concatenate(n_ccother_list)
n_ccinc = np.concatenate(n_ccinc_list)

print(f"NuWro (10 flat-tree files): {len(n_q2)} rows, "
      f"CC-inclusive={n_ccinc.sum()}, CC-other={n_ccother.sum()}")
print(f"  W_nuc_rest NaN frac within CC-inclusive: {np.mean(~np.isfinite(n_w[n_ccinc])):.5f}")


## Fraction-per-bin and the GENIE/NuWro ratio, conditioned on a W selection

For each generator and each W selection, `f(Q2 bin) = (CC-other AND in this Q2 bin AND passes the W cut) / (all CC-inclusive events passing the same W cut)`. Both numerator and denominator are restricted to the same W selection, so each curve is "what fraction of CC-inclusive-and-this-W-regime interactions are CC-other, as a function of Q2" -- not a Q2 shape re-weighted by a global W cut applied only to the numerator.

Uncertainty on each fraction uses the standard binomial-count formula $\sigma_f = \sqrt{f(1-f)/n}$ (a fraction of a fixed total, not $\sqrt{N}$), and the ratio uncertainty combines the two fractions' relative uncertainties in quadrature.

In [ ]:
def ccother_fraction_per_bin(q2, is_ccother, is_ccinc, w, w_mask_fn, edges):
    wsel = w_mask_fn(w) if w_mask_fn is not None else np.ones_like(w, dtype=bool)
    denom_mask = is_ccinc & wsel & np.isfinite(w)
    q2_ccinc = q2[denom_mask]
    other_flag = is_ccother[denom_mask]
    n_total = denom_mask.sum()
    fracs, ns_other = [], []
    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i + 1]
        m = other_flag & (q2_ccinc >= lo) & (q2_ccinc < hi)
        k = m.sum()
        fracs.append(k / n_total if n_total > 0 else np.nan)
        ns_other.append(k)
    return np.array(fracs), np.array(ns_other), n_total

W_SELECTIONS = {
    'All W': None,
    'W < 1.4 GeV': (lambda w: w < W_SPLIT),
    'W > 1.4 GeV': (lambda w: w >= W_SPLIT),
}

results = {}
for label, wfn in W_SELECTIONS.items():
    f_g, k_g, n_g = ccother_fraction_per_bin(g_q2, g_ccother, g_ccinc, g_w, wfn, Q2_EDGES)
    f_n, k_n, n_n = ccother_fraction_per_bin(n_q2, n_ccother, n_ccinc, n_w, wfn, Q2_EDGES)

    sig_fg = np.sqrt(f_g * (1 - f_g) / n_g)
    sig_fn = np.sqrt(f_n * (1 - f_n) / n_n)

    R = f_g / f_n
    sig_R = R * np.sqrt((sig_fg / f_g) ** 2 + (sig_fn / f_n) ** 2)

    results[label] = dict(f_g=f_g, f_n=f_n, k_g=k_g, n_g=n_g, R=R, sig_R=sig_R)

    print(f"=== {label} ===  (denom: GENIE n={n_g}, NuWro n={n_n})")
    print(f"{'Q2 bin':>12} | {'f_GENIE':>9} | {'f_NuWro':>9} | {'R=fG/fN':>8} | {'sigR':>7}")
    for i in range(len(Q2_EDGES) - 1):
        lo, hi = Q2_EDGES[i], Q2_EDGES[i + 1]
        print(f"[{lo:.1f},{hi:.1f}) | {f_g[i]:9.5f} | {f_n[i]:9.5f} | {R[i]:8.3f} | {sig_R[i]:7.3f}")
    print()


In [ ]:
centers = 0.5 * (Q2_EDGES[:-1] + Q2_EDGES[1:])
colors = {'All W': 'tab:blue', 'W < 1.4 GeV': 'tab:orange', 'W > 1.4 GeV': 'tab:green'}

fig, ax = plt.subplots(figsize=(6, 5))
for label in ['All W', 'W < 1.4 GeV', 'W > 1.4 GeV']:
    R = results[label]['R']
    sig = results[label]['sig_R']
    valid = np.isfinite(sig) & (R > 0)   # drop the highest Q2 bin(s) where GENIE has 0 events -- R=0/undefined, not a real ratio
    ax.errorbar(centers[valid], R[valid], yerr=sig[valid], fmt='o', markersize=5,
                color=colors[label], label=label, capsize=2)

ax.axhline(1.0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel(r'$Q^2$ (GeV$^2$)', fontsize=12)
ax.set_ylabel('GENIE/NuWro', fontsize=12)
ax.set_xlim(0, 10)
ax.set_ylim(0.0, 1.6)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.25)
fig.tight_layout()

import os
os.makedirs('genie_nuwro_plots', exist_ok=True)
fig.savefig('genie_nuwro_plots/genie_nuwro_ratio_ourfile.pdf', bbox_inches='tight')
fig.savefig('genie_nuwro_plots/genie_nuwro_ratio_ourfile.png', bbox_inches='tight', dpi=150)


## Interpretation: do we get the exact same plot?

**No -- not even close in shape, not just in exact numbers.** Howard's plot has all three curves starting above 1 at low $Q^2$ (roughly 1.3-1.45), decreasing with $Q^2$, and crossing below 1 somewhere around $Q^2 \sim 5$-$6\,\mathrm{GeV}^2$, ending around 0.65-1.0 at $Q^2=10$. Ours instead starts around 0.33-0.71 at low $Q^2$ and falls monotonically to below 0.1 by $Q^2 \sim 8$-$9\,\mathrm{GeV}^2$, **never crossing above 1 anywhere, in any of the three W selections.**

This is exactly the population-mismatch effect flagged in the caveat at the top, now visible in the numbers rather than just asserted: our own file's `selected`/`sideband`/`signal` trees require reconstructing a candidate topology before an event appears at all, while NuWro's `flagCCINC` count includes every generated CC interaction regardless of whether anything reconstructable came out of it. Since a true CC-other (pion-production/multi-nucleon/etc.) event is disproportionately likely to fail that reconstruction requirement relative to a "clean" CC0$\pi$-like event, our GENIE numerator and denominator are both suppressed relative to NuWro's bare-truth count, and the CC-other fraction ends up systematically smaller at every $Q^2$ -- not because GENIE and NuWro disagree on the physics by that much, but because one side has selection efficiency baked in and the other doesn't.

So this notebook does not answer "do GENIE and NuWro agree the way Howard found them to" -- it answers "what does our own reconstructed background population's Q2/W composition look like, relative to NuWro's bare generator truth," which is a different, though not meaningless, question. To actually check whether we reproduce Howard's result, the GENIE side would need to come from the equivalent bare NUISANCE flat trees (`data/Generators/GENIE/fhc_Nu14/`), the same way NuWro is treated here -- that comparison hasn't been run yet.